# S5 Contrastive SSL Colab

This notebook builds a Colab-friendly S5 self-supervised learning baseline on Utah-array cache shards stored in Google Drive.

It supports two contrastive objectives behind one switch:

- `future_infonce`: CPC-style future-state InfoNCE over S5 hidden states.
- `augment_infonce`: same-window augmentation contrastive learning over pooled segment embeddings.

The notebook keeps the existing `s5_future_prediction.ipynb` data-loading pattern:

- mount Google Drive
- clone the public repo
- copy cache data to `/content`
- train from local disk
- save logs, checkpoints, and plots back to Drive


In [1]:
# Mount Drive and resolve cache / output roots.
from google.colab import drive
from pathlib import Path


drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive")
cache_candidates = [
    DRIVE_ROOT / "utah_ssl" / "data" / "cache_v1",
    DRIVE_ROOT / "utah_ssl" / "data" / "cache_v1_fused",
    DRIVE_ROOT / "utah_ssl" / "data" / "v1_cache",
    DRIVE_ROOT / "utah_ssl" / "data" / "v1_cache_fused",
    DRIVE_ROOT / "cache_v1",
    DRIVE_ROOT / "cache_v1_fused",
    DRIVE_ROOT / "v1_cache",
    DRIVE_ROOT / "v1_cache_fused",
]
CACHE_ROOT = next((p for p in cache_candidates if p.exists()), cache_candidates[0])
OUTPUT_ROOT = DRIVE_ROOT / "utah_ssl" / "outputs" / "ssl_experiments" / "contrastive"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("DRIVE_ROOT :", DRIVE_ROOT)
print("CACHE_ROOT :", CACHE_ROOT, "| exists:", CACHE_ROOT.exists())
print("OUTPUT_ROOT:", OUTPUT_ROOT, "| exists:", OUTPUT_ROOT.exists())

if CACHE_ROOT.exists():
    datasets = sorted(p.name for p in CACHE_ROOT.iterdir() if p.is_dir())
    print("datasets:", datasets)
else:
    print("cache candidates checked:")
    for path in cache_candidates:
        print(" -", path)


In [2]:
# Clone the public repo and import the canonical S5 reference block.
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/ethan-read/utah-ssl.git"
REPO_DIR = Path("/content/utah-ssl")
SSL_DIR = REPO_DIR / "analysis" / "active" / "transfer_benchmark" / "ssl_autoresearch"

if REPO_DIR.exists():
    print("Using existing repo:", REPO_DIR)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)

for path in (REPO_DIR, SSL_DIR):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

os.environ["SSL_AUTORESEARCH_CACHE_ROOT"] = str(CACHE_ROOT)
os.environ["SSL_AUTORESEARCH_OUTPUT_ROOT"] = str(OUTPUT_ROOT)

from s5 import S5SequenceBackbone

benchmark_split_helpers = None
try:
    from data import split_latest_sessions, session_ids_from_cache_split
    benchmark_split_helpers = {
        "split_latest_sessions": split_latest_sessions.__name__,
        "session_ids_from_cache_split": session_ids_from_cache_split.__name__,
    }
except Exception as exc:
    benchmark_split_helpers = {"import_error": repr(exc)}

print("cwd:", Path.cwd())
print("ssl dir exists:", SSL_DIR.exists(), SSL_DIR)
print("benchmark split helper status:", benchmark_split_helpers)
print("SSL_AUTORESEARCH_CACHE_ROOT:", os.environ["SSL_AUTORESEARCH_CACHE_ROOT"])
print("SSL_AUTORESEARCH_OUTPUT_ROOT:", os.environ["SSL_AUTORESEARCH_OUTPUT_ROOT"])


In [3]:
# Experiment config.
SEED = 7
OBJECTIVE_MODE = "future_infonce"
SEGMENT_BINS = 64
FUTURE_HORIZONS = (1, 3)
PATCH_SIZE = 1
PATCH_STRIDE = 1
HIDDEN_SIZE = 128
S5_STATE_SIZE = 64
NUM_LAYERS = 1
DROPOUT = 0.1
BATCH_SIZE = 32
NUM_STEPS = 500
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-2
TEMPERATURE = 0.1
VAL_EVERY = 50
VAL_BATCHES = 10
DATASET_WEIGHT_ALPHA = 0.25
EXAMPLES_PER_SHARD = 8
POST_PROJ_NORM = "rms"
EXCLUDED_DATASETS = {"brain2text25"}
LOG_EVERY = 10

AUGMENT_CFG = {
    "noise_std": 0.01,
    "scale_jitter": 0.05,
    "offset_jitter": 0.05,
    "time_mask_frac": 0.10,
    "channel_dropout_prob": 0.05,
    "clip_value": 20.0,
}

assert OBJECTIVE_MODE in {"future_infonce", "augment_infonce"}
assert PATCH_STRIDE <= PATCH_SIZE

print("OBJECTIVE_MODE:", OBJECTIVE_MODE)
print("SEGMENT_BINS:", SEGMENT_BINS)
print("FUTURE_HORIZONS:", FUTURE_HORIZONS)
print("PATCH:", (PATCH_SIZE, PATCH_STRIDE))
print("MODEL:", {"hidden_size": HIDDEN_SIZE, "s5_state_size": S5_STATE_SIZE, "num_layers": NUM_LAYERS, "dropout": DROPOUT})
print("TRAIN:", {"batch_size": BATCH_SIZE, "num_steps": NUM_STEPS, "lr": LEARNING_RATE, "weight_decay": WEIGHT_DECAY, "temperature": TEMPERATURE})
print("VAL:", {"val_every": VAL_EVERY, "val_batches": VAL_BATCHES})
print("AUGMENT_CFG:", AUGMENT_CFG)


In [4]:
# Copy cache to local storage, summarize datasets, split sessions, and build samplers.
import hashlib
import json
import math
import os
import random
import shutil
import time
from collections import Counter, OrderedDict, defaultdict
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import torch

try:
    import psutil
except ImportError:
    psutil = None

TX_DIM = 256
SBP_DIM = 256
FULL_DIM = TX_DIM + SBP_DIM
NORMALIZE_IMPL_VERSION = "segment_prefix_v1"
NORMALIZE_CONTEXT_BINS = min(16, SEGMENT_BINS)
DEFAULT_EXAMPLES_PER_SHARD = EXAMPLES_PER_SHARD

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DRIVE_CACHE_ROOT = next((p for p in cache_candidates if p.exists()), cache_candidates[0])
LOCAL_CACHE_ROOT = Path("/content/utah_ssl_cache") / DRIVE_CACHE_ROOT.name
LOCAL_CACHE_STATUS_PATH = LOCAL_CACHE_ROOT.parent / f"{DRIVE_CACHE_ROOT.name}_copy_status.json"
FORCE_RECOPY_LOCAL_CACHE = False

if not DRIVE_CACHE_ROOT.exists():
    raise FileNotFoundError(f"Drive cache root not found: {DRIVE_CACHE_ROOT}")

if FORCE_RECOPY_LOCAL_CACHE and LOCAL_CACHE_ROOT.exists():
    print("removing existing local cache:", LOCAL_CACHE_ROOT)
    shutil.rmtree(LOCAL_CACHE_ROOT)
if FORCE_RECOPY_LOCAL_CACHE and LOCAL_CACHE_STATUS_PATH.exists():
    LOCAL_CACHE_STATUS_PATH.unlink()


def _format_bytes(num_bytes: int) -> str:
    value = float(num_bytes)
    for unit in ["B", "KB", "MB", "GB", "TB"]:
        if value < 1024.0 or unit == "TB":
            return f"{value:.1f} {unit}"
        value /= 1024.0


def _path_signature(path: Path) -> dict | None:
    if not path.exists():
        return None
    stat = path.stat()
    return {
        "size": int(stat.st_size),
        "mtime_ns": int(stat.st_mtime_ns),
    }


def _compute_cache_source_signature(src_root: Path) -> str:
    datasets = []
    for dataset_root in sorted(p for p in src_root.iterdir() if p.is_dir()):
        shard_root = dataset_root / "shards"
        shard_names = sorted(p.name for p in shard_root.iterdir() if p.is_dir()) if shard_root.exists() else []
        datasets.append(
            {
                "dataset": dataset_root.name,
                "manifest": _path_signature(dataset_root / "manifest.jsonl"),
                "metadata": _path_signature(dataset_root / "metadata.json"),
                "shard_count": len(shard_names),
                "first_shard": shard_names[0] if shard_names else None,
                "last_shard": shard_names[-1] if shard_names else None,
            }
        )
    payload = {
        "root": str(src_root),
        "datasets": datasets,
        "repack_summary": _path_signature(src_root / "repack_summary.json"),
    }
    return hashlib.sha256(json.dumps(payload, sort_keys=True).encode("utf-8")).hexdigest()


SOURCE_CACHE_SIGNATURE = _compute_cache_source_signature(DRIVE_CACHE_ROOT)


def _load_copy_status() -> dict | None:
    if not LOCAL_CACHE_STATUS_PATH.exists():
        return None
    try:
        return json.loads(LOCAL_CACHE_STATUS_PATH.read_text())
    except Exception:
        return None


def _copy_complete_for_current_source(status: dict | None) -> bool:
    return bool(
        status
        and status.get("complete") is True
        and status.get("source") == str(DRIVE_CACHE_ROOT)
        and status.get("source_signature") == SOURCE_CACHE_SIGNATURE
        and Path(status.get("dest", str(LOCAL_CACHE_ROOT))).exists()
    )


def _write_copy_status(*, file_count: int, total_bytes: int) -> None:
    payload = {
        "complete": True,
        "source": str(DRIVE_CACHE_ROOT),
        "source_signature": SOURCE_CACHE_SIGNATURE,
        "dest": str(LOCAL_CACHE_ROOT),
        "file_count": int(file_count),
        "total_bytes": int(total_bytes),
        "written_at": time.time(),
    }
    LOCAL_CACHE_STATUS_PATH.write_text(json.dumps(payload, indent=2))


def copy_tree_with_progress(
    src_root: Path,
    dst_root: Path,
    *,
    print_every_files: int = 250,
    max_copy_retries: int = 5,
) -> tuple[int, int]:
    entries = sorted(src_root.iterdir(), key=lambda path: path.name)
    file_count = 0
    total_bytes = 0
    for path in entries:
        if path.is_file():
            file_count += 1
            total_bytes += path.stat().st_size
            continue
        for child in path.rglob("*"):
            if child.is_file():
                file_count += 1
                total_bytes += child.stat().st_size

    print(
        f"copy plan: {len(entries)} top-level entries, {file_count} files, {_format_bytes(total_bytes)} total"
    )

    copied_files = 0
    copied_bytes = 0
    last_report = time.time()
    start_time = last_report

    def report(force: bool = False, label: str | None = None) -> None:
        nonlocal last_report
        now = time.time()
        if not force and copied_files > 0 and copied_files % print_every_files != 0 and (now - last_report) < 15.0:
            return
        elapsed = max(now - start_time, 1e-6)
        rate = copied_bytes / elapsed
        prefix = "progress"
        if label is not None:
            prefix = f"progress [{label}]"
        print(
            f"{prefix}: files={copied_files}/{file_count} bytes={_format_bytes(copied_bytes)}/{_format_bytes(total_bytes)} rate={_format_bytes(int(rate))}/s elapsed={elapsed:.1f}s"
        )
        last_report = now

    def copy_path(src_path: Path, dst_path: Path, *, label: str) -> None:
        nonlocal copied_files, copied_bytes
        if src_path.is_dir():
            dst_path.mkdir(parents=True, exist_ok=True)
            for child in sorted(src_path.iterdir(), key=lambda path: path.name):
                copy_path(child, dst_path / child.name, label=label)
            return

        src_size = src_path.stat().st_size
        if dst_path.exists() and dst_path.stat().st_size == src_size:
            copied_files += 1
            copied_bytes += src_size
            report(label=label)
            return

        dst_path.parent.mkdir(parents=True, exist_ok=True)
        last_error = None
        for attempt in range(1, max_copy_retries + 1):
            try:
                shutil.copy2(src_path, dst_path)
                copied_files += 1
                copied_bytes += src_size
                report(label=label)
                return
            except OSError as exc:
                last_error = exc
                print(f"copy retry {attempt}/{max_copy_retries} failed for {src_path}: {exc}")
                time.sleep(min(10, 2 * attempt))

        raise OSError(f"Failed to copy {src_path} after {max_copy_retries} retries") from last_error

    dst_root.mkdir(parents=True, exist_ok=True)
    for entry_idx, src_path in enumerate(entries, start=1):
        label = f"{entry_idx}/{len(entries)} {src_path.name}"
        print(f"starting {label}")
        copy_path(src_path, dst_root / src_path.name, label=label)
        report(force=True, label=label)

    return file_count, total_bytes


copy_status = _load_copy_status()
if (not LOCAL_CACHE_ROOT.exists()) or (not _copy_complete_for_current_source(copy_status)):
    if LOCAL_CACHE_ROOT.exists():
        print("removing stale local cache:", LOCAL_CACHE_ROOT)
        shutil.rmtree(LOCAL_CACHE_ROOT)
    if LOCAL_CACHE_STATUS_PATH.exists():
        LOCAL_CACHE_STATUS_PATH.unlink()
    LOCAL_CACHE_ROOT.parent.mkdir(parents=True, exist_ok=True)
    action = "copying"
    print(f"{action} cache to local disk...")
    print("source:", DRIVE_CACHE_ROOT)
    print("source signature:", SOURCE_CACHE_SIGNATURE[:12])
    print("dest  :", LOCAL_CACHE_ROOT)
    t0 = time.time()
    file_count, total_bytes = copy_tree_with_progress(DRIVE_CACHE_ROOT, LOCAL_CACHE_ROOT)
    _write_copy_status(file_count=file_count, total_bytes=total_bytes)
    print(f"copy complete in {time.time() - t0:.1f}s")
else:
    print("using existing local cache:", LOCAL_CACHE_ROOT)
    print("source signature:", SOURCE_CACHE_SIGNATURE[:12])

CACHE_ROOT = LOCAL_CACHE_ROOT
os.environ["SSL_AUTORESEARCH_CACHE_ROOT"] = str(CACHE_ROOT)

dataset_roots = sorted(p for p in Path(CACHE_ROOT).iterdir() if p.is_dir())
available_datasets = [p.name for p in dataset_roots]
pretrain_datasets = [name for name in available_datasets if name not in EXCLUDED_DATASETS]

print("available datasets:")
for name in available_datasets:
    tag = "excluded" if name in EXCLUDED_DATASETS else "included"
    print(f" - {name} [{tag}]")


@dataclass(frozen=True)
class ExampleRow:
    dataset: str
    session_id: str
    shard_relpath: str
    example_index: int
    n_time_bins: int
    has_tx: bool
    has_sbp: bool
    n_tx_features: int
    n_sbp_features: int


@dataclass(frozen=True)
class SamplingPlan:
    split_name: str
    segment_bins: int
    dataset_weight_alpha: float
    dataset_names: tuple
    dataset_probs: np.ndarray
    shard_rows_by_dataset: dict
    shard_keys_by_dataset: dict
    shard_probs_by_dataset: dict
    row_probs_within_shard_by_dataset: dict


def stable_text_seed(text: str, base_seed: int) -> int:
    return int(base_seed + sum((idx + 1) * ord(ch) for idx, ch in enumerate(text)))


def _choose_shard_cache_gb() -> float:
    if psutil is None:
        return 4.0
    available_gb = psutil.virtual_memory().available / (1024 ** 3)
    return float(min(8.0, max(2.0, 0.35 * available_gb)))


SHARD_CACHE_RAM_GB = round(_choose_shard_cache_gb(), 2)
rows_by_dataset = {}
split_rows_by_dataset = {"train": {}, "val": {}}
session_split_summary = {}

for dataset in pretrain_datasets:
    ds_root = Path(CACHE_ROOT) / dataset
    manifest_path = ds_root / "manifest.jsonl"
    if not manifest_path.exists():
        raise FileNotFoundError(f"Manifest missing for dataset {dataset}: {manifest_path}")

    rows = []
    with manifest_path.open() as handle:
        for line in handle:
            payload = json.loads(line)
            rows.append(
                ExampleRow(
                    dataset=dataset,
                    session_id=str(payload["session_id"]),
                    shard_relpath=str(payload["shard_relpath"]),
                    example_index=int(payload["example_index"]),
                    n_time_bins=int(payload["n_time_bins"]),
                    has_tx=bool(payload.get("has_tx", False)),
                    has_sbp=bool(payload.get("has_sbp", False)),
                    n_tx_features=int(payload.get("n_tx_features", 0) or 0),
                    n_sbp_features=int(payload.get("n_sbp_features", 0) or 0),
                )
            )
    rows_by_dataset[dataset] = rows

    session_ids = sorted({row.session_id for row in rows})
    if len(session_ids) < 2:
        train_session_ids = list(session_ids)
        val_session_ids = []
    else:
        split_rng = random.Random(stable_text_seed(dataset, SEED))
        shuffled = list(session_ids)
        split_rng.shuffle(shuffled)
        val_count = max(1, int(math.ceil(0.2 * len(shuffled))))
        val_count = min(val_count, len(shuffled) - 1)
        val_session_ids = sorted(shuffled[:val_count])
        train_session_ids = sorted(shuffled[val_count:])

    train_set = set(train_session_ids)
    val_set = set(val_session_ids)
    split_rows_by_dataset["train"][dataset] = [row for row in rows if row.session_id in train_set]
    split_rows_by_dataset["val"][dataset] = [row for row in rows if row.session_id in val_set]
    session_split_summary[dataset] = {
        "total_sessions": len(session_ids),
        "train_sessions": len(train_session_ids),
        "val_sessions": len(val_session_ids),
        "val_eligible": len(session_ids) >= 2,
        "train_examples": len(split_rows_by_dataset["train"][dataset]),
        "val_examples": len(split_rows_by_dataset["val"][dataset]),
    }

print("\npretrain datasets:", pretrain_datasets)
print("NORMALIZE_IMPL_VERSION:", NORMALIZE_IMPL_VERSION)
print("NORMALIZE_CONTEXT_BINS:", NORMALIZE_CONTEXT_BINS)
print("SHARD_CACHE_RAM_GB:", SHARD_CACHE_RAM_GB)
print("\nsession split summary:")
for dataset in pretrain_datasets:
    summary = session_split_summary[dataset]
    print(
        f" - {dataset}: sessions={summary['total_sessions']} train_sessions={summary['train_sessions']} "
        f"val_sessions={summary['val_sessions']} train_examples={summary['train_examples']} "
        f"val_examples={summary['val_examples']} val_eligible={summary['val_eligible']}"
    )


class ShardStore:
    def __init__(self, cache_root: Path, ram_cache_gb: float):
        self.cache_root = Path(cache_root)
        self.max_bytes = int(ram_cache_gb * (1024 ** 3))
        self._cache = OrderedDict()
        self._cached_bytes = 0

    def clear(self):
        self._cache.clear()
        self._cached_bytes = 0

    def summary(self):
        return {
            "cached_shards": len(self._cache),
            "cached_gb": self._cached_bytes / (1024 ** 3),
            "budget_gb": self.max_bytes / (1024 ** 3),
        }

    def _load_array(self, path: Path):
        if not path.exists():
            return None
        return np.load(path, mmap_mode="r", allow_pickle=False)

    def _load_shard(self, shard_relpath: str):
        shard_path = self.cache_root / shard_relpath
        shard = {
            "time_offsets": self._load_array(shard_path / "time_offsets.npy"),
            "tx": self._load_array(shard_path / "tx.npy"),
            "sbp": self._load_array(shard_path / "sbp.npy"),
        }
        shard["bytes"] = int(
            shard["time_offsets"].nbytes
            + (0 if shard["tx"] is None else shard["tx"].nbytes)
            + (0 if shard["sbp"] is None else shard["sbp"].nbytes)
        )
        return shard

    def get(self, shard_relpath: str):
        key = str(shard_relpath)
        cached = self._cache.get(key)
        if cached is not None:
            self._cache.move_to_end(key)
            return cached

        shard = self._load_shard(key)
        shard_bytes = shard["bytes"]
        if shard_bytes <= self.max_bytes:
            while self._cache and self._cached_bytes + shard_bytes > self.max_bytes:
                _, evicted = self._cache.popitem(last=False)
                self._cached_bytes -= evicted["bytes"]
            self._cache[key] = shard
            self._cached_bytes += shard_bytes
        return shard


SHARD_STORE = ShardStore(CACHE_ROOT, SHARD_CACHE_RAM_GB)
print("\nshard cache:", SHARD_STORE.summary())


def _normalize_segment(
    x_seq: torch.Tensor,
    feature_mask: torch.Tensor,
    context_bins: int,
    *,
    min_scale_std: float = 0.1,
    clip_value: float = 20.0,
) -> torch.Tensor:
    x_norm = x_seq.clone()
    present_idx = torch.nonzero(feature_mask.bool(), as_tuple=False).squeeze(1)
    if present_idx.numel() == 0:
        return x_norm

    context_x = x_norm[:context_bins, present_idx]
    mean = context_x.mean(dim=0)
    centered = x_norm[:, present_idx] - mean
    std = context_x.std(dim=0, unbiased=False)
    scale_mask = std >= min_scale_std
    if scale_mask.any():
        centered[:, scale_mask] = centered[:, scale_mask] / std[scale_mask]
    x_norm[:, present_idx] = centered.clamp(min=-clip_value, max=clip_value)
    return x_norm


def sample_base_segment(example: ExampleRow, segment_bins: int, py_rng: random.Random):
    shard = SHARD_STORE.get(example.shard_relpath)
    time_offsets = shard["time_offsets"]
    start = int(time_offsets[example.example_index])
    stop = int(time_offsets[example.example_index + 1])
    length = stop - start
    total_needed = int(segment_bins)
    max_start = length - total_needed
    if max_start < 0:
        raise ValueError(
            f"Example {example.dataset}:{example.session_id} length={length} cannot support segment_bins={segment_bins}"
        )

    offset = py_rng.randrange(max_start + 1)
    src_start = start + offset
    src_stop = src_start + total_needed

    x_seq = np.zeros((total_needed, FULL_DIM), dtype=np.float32)
    feature_mask = np.zeros((FULL_DIM,), dtype=np.float32)

    if shard["tx"] is not None:
        tx = np.asarray(shard["tx"][src_start:src_stop], dtype=np.float32)
        tx_dim = min(tx.shape[1], TX_DIM)
        x_seq[:, :tx_dim] = tx[:, :tx_dim]
        feature_mask[:tx_dim] = 1.0

    if shard["sbp"] is not None:
        sbp = np.asarray(shard["sbp"][src_start:src_stop], dtype=np.float32)
        sbp_dim = min(sbp.shape[1], SBP_DIM)
        x_seq[:, TX_DIM:TX_DIM + sbp_dim] = sbp[:, :sbp_dim]
        feature_mask[TX_DIM:TX_DIM + sbp_dim] = 1.0

    x_seq_t = torch.from_numpy(x_seq)
    feature_mask_t = torch.from_numpy(feature_mask)
    x_norm = _normalize_segment(x_seq_t, feature_mask_t, context_bins=NORMALIZE_CONTEXT_BINS)

    return {
        "x": x_norm,
        "feature_mask": feature_mask_t,
        "length": int(segment_bins),
        "dataset": example.dataset,
        "session_id": example.session_id,
        "session_key": f"{example.dataset}:{example.session_id}",
        "shard_relpath": example.shard_relpath,
        "has_tx": example.has_tx,
        "has_sbp": example.has_sbp,
        "orig_len": length,
    }


def stack_segment_batch(samples):
    return {
        "x": torch.stack([item["x"] for item in samples], dim=0),
        "feature_mask": torch.stack([item["feature_mask"] for item in samples], dim=0),
        "lengths": torch.tensor([item["length"] for item in samples], dtype=torch.long),
        "datasets": [item["dataset"] for item in samples],
        "session_keys": [item["session_key"] for item in samples],
        "sessions": [item["session_id"] for item in samples],
        "shard_relpaths": [item["shard_relpath"] for item in samples],
    }


def _valid_row_weights(rows, segment_bins: int) -> np.ndarray:
    return np.array([max(0, row.n_time_bins - segment_bins + 1) for row in rows], dtype=np.float64)


_sampling_plan_cache = {}


def get_sampling_plan(
    split_name: str,
    segment_bins: int,
    dataset_weight_alpha: float = DATASET_WEIGHT_ALPHA,
):
    key = (split_name, int(segment_bins), float(dataset_weight_alpha))
    cached = _sampling_plan_cache.get(key)
    if cached is not None:
        return cached

    shard_rows_by_dataset = {}
    shard_keys_by_dataset = {}
    shard_probs_by_dataset = {}
    row_probs_within_shard_by_dataset = {}
    dataset_mass = {}

    for dataset in pretrain_datasets:
        rows = split_rows_by_dataset[split_name][dataset]
        weights = _valid_row_weights(rows, segment_bins)
        keep_mask = weights > 0
        kept_rows = [row for row, keep in zip(rows, keep_mask) if keep]
        kept_weights = weights[keep_mask]
        if len(kept_rows) == 0:
            continue

        dataset_mass[dataset] = float(kept_weights.sum())
        shard_rows = defaultdict(list)
        shard_weights = defaultdict(list)
        for row, weight in zip(kept_rows, kept_weights):
            shard_rows[row.shard_relpath].append(row)
            shard_weights[row.shard_relpath].append(float(weight))

        shard_keys = list(shard_rows.keys())
        shard_mass = np.array([sum(shard_weights[key_name]) for key_name in shard_keys], dtype=np.float64)
        shard_probs = shard_mass / shard_mass.sum()

        shard_rows_by_dataset[dataset] = dict(shard_rows)
        shard_keys_by_dataset[dataset] = shard_keys
        shard_probs_by_dataset[dataset] = shard_probs
        row_probs_within_shard_by_dataset[dataset] = {
            key_name: np.array(weight_list, dtype=np.float64) / np.sum(weight_list)
            for key_name, weight_list in shard_weights.items()
        }

    dataset_names = tuple(dataset for dataset in pretrain_datasets if dataset in dataset_mass)
    if not dataset_names:
        raise RuntimeError(f"Split {split_name} has no datasets with enough bins for segment_bins={segment_bins}")

    dataset_probs = np.array(
        [dataset_mass[dataset] ** dataset_weight_alpha for dataset in dataset_names],
        dtype=np.float64,
    )
    dataset_probs = dataset_probs / dataset_probs.sum()

    plan = SamplingPlan(
        split_name=split_name,
        segment_bins=int(segment_bins),
        dataset_weight_alpha=float(dataset_weight_alpha),
        dataset_names=dataset_names,
        dataset_probs=dataset_probs,
        shard_rows_by_dataset=shard_rows_by_dataset,
        shard_keys_by_dataset=shard_keys_by_dataset,
        shard_probs_by_dataset=shard_probs_by_dataset,
        row_probs_within_shard_by_dataset=row_probs_within_shard_by_dataset,
    )
    _sampling_plan_cache[key] = plan
    return plan


class SegmentBatchSampler:
    def __init__(
        self,
        split_name: str,
        segment_bins: int,
        batch_size: int,
        seed: int = SEED,
        dataset_weight_alpha: float = DATASET_WEIGHT_ALPHA,
        examples_per_shard: int = DEFAULT_EXAMPLES_PER_SHARD,
    ):
        self.split_name = split_name
        self.segment_bins = int(segment_bins)
        self.batch_size = int(batch_size)
        self.examples_per_shard = max(1, int(examples_per_shard))
        self.seed = int(seed)
        self.plan = get_sampling_plan(split_name, self.segment_bins, dataset_weight_alpha)
        self.py_rng = random.Random(self.seed)
        self.np_rng = np.random.default_rng(self.seed)

    def sample_batch(self, batch_size: int | None = None):
        batch_size = self.batch_size if batch_size is None else int(batch_size)
        requested_dataset_idx = self.np_rng.choice(
            len(self.plan.dataset_names),
            size=batch_size,
            p=self.plan.dataset_probs,
        )
        dataset_counts = Counter(self.plan.dataset_names[int(idx)] for idx in requested_dataset_idx)

        samples = []
        for dataset, n_examples in dataset_counts.items():
            shard_keys = self.plan.shard_keys_by_dataset[dataset]
            shard_probs = self.plan.shard_probs_by_dataset[dataset]
            n_shards = max(1, math.ceil(n_examples / self.examples_per_shard))
            replace_shards = n_shards > len(shard_keys)
            sampled_shard_idx = self.np_rng.choice(
                len(shard_keys),
                size=n_shards,
                replace=replace_shards,
                p=shard_probs,
            )

            remaining = int(n_examples)
            for shard_choice_idx, shard_idx in enumerate(np.atleast_1d(sampled_shard_idx)):
                take = min(self.examples_per_shard, remaining)
                if shard_choice_idx == n_shards - 1:
                    take = remaining

                shard_key = shard_keys[int(shard_idx)]
                shard_rows = self.plan.shard_rows_by_dataset[dataset][shard_key]
                row_probs = self.plan.row_probs_within_shard_by_dataset[dataset][shard_key]
                row_choices = self.np_rng.choice(len(shard_rows), size=take, replace=True, p=row_probs)
                for row_idx in np.atleast_1d(row_choices):
                    example = shard_rows[int(row_idx)]
                    samples.append(sample_base_segment(example, segment_bins=self.segment_bins, py_rng=self.py_rng))

                remaining -= take
                if remaining <= 0:
                    break

        order = self.np_rng.permutation(len(samples))
        samples = [samples[int(idx)] for idx in order]
        return stack_segment_batch(samples)


HAS_VAL_DATASETS = any(
    session_split_summary[dataset]["val_examples"] > 0
    for dataset in pretrain_datasets
)


def build_segment_sampler(
    split_name: str,
    batch_size: int,
    seed: int = SEED,
    segment_bins: int = SEGMENT_BINS,
    dataset_weight_alpha: float = DATASET_WEIGHT_ALPHA,
    examples_per_shard: int = DEFAULT_EXAMPLES_PER_SHARD,
):
    if split_name == "val" and not HAS_VAL_DATASETS:
        raise RuntimeError("No validation datasets are eligible for session-disjoint validation.")
    return SegmentBatchSampler(
        split_name=split_name,
        segment_bins=segment_bins,
        batch_size=batch_size,
        seed=seed,
        dataset_weight_alpha=dataset_weight_alpha,
        examples_per_shard=examples_per_shard,
    )


preview_sampler = build_segment_sampler(
    "train",
    batch_size=min(4, BATCH_SIZE),
    seed=SEED,
    segment_bins=SEGMENT_BINS,
    dataset_weight_alpha=DATASET_WEIGHT_ALPHA,
    examples_per_shard=EXAMPLES_PER_SHARD,
)
preview_batch = preview_sampler.sample_batch()

print("\npreview batch:")
print(" - x           :", tuple(preview_batch["x"].shape))
print(" - feature_mask:", tuple(preview_batch["feature_mask"].shape))
print(" - lengths     :", tuple(preview_batch["lengths"].shape))
print(" - datasets    :", Counter(preview_batch["datasets"]))
print(" - shard cache :", SHARD_STORE.summary())


In [5]:
# Sampler smoke test.
INSPECT_BATCH_SIZE = 8
inspect_train_sampler = build_segment_sampler(
    "train",
    batch_size=INSPECT_BATCH_SIZE,
    seed=SEED,
    segment_bins=SEGMENT_BINS,
    dataset_weight_alpha=DATASET_WEIGHT_ALPHA,
    examples_per_shard=EXAMPLES_PER_SHARD,
)
inspect_batch = inspect_train_sampler.sample_batch()

print("train batch shapes:")
print(" - x           :", tuple(inspect_batch["x"].shape))
print(" - feature_mask:", tuple(inspect_batch["feature_mask"].shape))
print(" - lengths     :", tuple(inspect_batch["lengths"].shape))
print(" - dataset mix :", Counter(inspect_batch["datasets"]))
print(" - session keys:", inspect_batch["session_keys"][:3])

if HAS_VAL_DATASETS:
    inspect_val_sampler = build_segment_sampler(
        "val",
        batch_size=min(INSPECT_BATCH_SIZE, 4),
        seed=SEED + 1,
        segment_bins=SEGMENT_BINS,
        dataset_weight_alpha=DATASET_WEIGHT_ALPHA,
        examples_per_shard=EXAMPLES_PER_SHARD,
    )
    inspect_val_batch = inspect_val_sampler.sample_batch()
    print("\nval batch shapes:")
    print(" - x           :", tuple(inspect_val_batch["x"].shape))
    print(" - feature_mask:", tuple(inspect_val_batch["feature_mask"].shape))
    print(" - lengths     :", tuple(inspect_val_batch["lengths"].shape))
    print(" - dataset mix :", Counter(inspect_val_batch["datasets"]))
else:
    print("\nNo validation sampler was built because all datasets are single-session.")


In [6]:
# Define the S5 encoder, contrastive losses, and smoke tests for both modes.
import torch.nn as nn
import torch.nn.functional as F

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)


def sync_device(device: torch.device) -> None:
    if device.type == "cuda":
        torch.cuda.synchronize(device)


class RMSNorm(nn.Module):
    def __init__(self, dim: int, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.scale = nn.Parameter(torch.ones(dim))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        rms = x.pow(2).mean(dim=-1, keepdim=True).add(self.eps).sqrt()
        return (x / rms) * self.scale


def _sequence_mask(lengths: torch.Tensor, max_len: int) -> torch.Tensor:
    positions = torch.arange(max_len, device=lengths.device).unsqueeze(0)
    return positions < lengths.unsqueeze(1)


def masked_mean_pool(hidden: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
    mask = _sequence_mask(lengths, hidden.shape[1]).unsqueeze(-1).to(hidden.dtype)
    denom = mask.sum(dim=1).clamp_min(1.0)
    return (hidden * mask).sum(dim=1) / denom


class ProjectionHead(nn.Module):
    def __init__(self, dim: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim),
            nn.GELU(),
            nn.Linear(dim, dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class S5ContrastiveEncoder(nn.Module):
    def __init__(
        self,
        *,
        input_dim: int = FULL_DIM,
        hidden_size: int = HIDDEN_SIZE,
        s5_state_size: int = S5_STATE_SIZE,
        num_layers: int = NUM_LAYERS,
        dropout: float = DROPOUT,
        patch_size: int = PATCH_SIZE,
        patch_stride: int = PATCH_STRIDE,
        post_proj_norm: str = POST_PROJ_NORM,
    ):
        super().__init__()
        self.input_dim = int(input_dim)
        self.hidden_size = int(hidden_size)
        self.patch_size = int(patch_size)
        self.patch_stride = int(patch_stride)
        self.token_dim = self.input_dim * self.patch_size
        self.proj = nn.Linear(self.token_dim, self.hidden_size)
        self.post_proj_norm = RMSNorm(self.hidden_size) if post_proj_norm == "rms" else nn.Identity()
        self.backbone = S5SequenceBackbone(
            d_model=self.hidden_size,
            d_state=int(s5_state_size),
            num_layers=int(num_layers),
            dropout=float(dropout),
            ffn_multiplier=2.0,
        )

    def _patch_one(self, sample: torch.Tensor, length: int) -> torch.Tensor:
        valid = sample[:length]
        if self.patch_size == 1:
            return valid

        starts = list(range(0, max(length - self.patch_size + 1, 1), self.patch_stride))
        patches = []
        for start in starts:
            patch = valid[start : start + self.patch_size]
            if patch.shape[0] < self.patch_size:
                pad = valid.new_zeros((self.patch_size - patch.shape[0], valid.shape[1]))
                patch = torch.cat([patch, pad], dim=0)
            patches.append(patch.reshape(-1))
        return torch.stack(patches, dim=0)

    def _patch_batch(self, x: torch.Tensor, lengths: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        token_sequences = []
        token_lengths = []
        for sample, length_tensor in zip(x, lengths):
            length = int(length_tensor.item())
            tokens = self._patch_one(sample, length)
            token_sequences.append(tokens)
            token_lengths.append(int(tokens.shape[0]))

        max_tokens = max(token_lengths)
        tokens = x.new_zeros((len(token_sequences), max_tokens, self.token_dim))
        for idx, token_sequence in enumerate(token_sequences):
            tokens[idx, : token_sequence.shape[0]] = token_sequence
        return tokens, torch.tensor(token_lengths, device=lengths.device, dtype=torch.long)

    def forward(self, x: torch.Tensor, lengths: torch.Tensor) -> dict[str, torch.Tensor]:
        tokens, token_lengths = self._patch_batch(x, lengths)
        hidden = self.post_proj_norm(self.proj(tokens))
        hidden = self.backbone(hidden, token_lengths)
        return {
            "hidden": hidden,
            "token_lengths": token_lengths,
            "tokens": tokens,
        }


class ContrastiveSSLModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = S5ContrastiveEncoder()
        self.anchor_head = ProjectionHead(HIDDEN_SIZE)
        self.future_head = ProjectionHead(HIDDEN_SIZE)
        self.segment_head = ProjectionHead(HIDDEN_SIZE)

    def encode_sequence(self, x: torch.Tensor, lengths: torch.Tensor) -> dict[str, torch.Tensor]:
        return self.encoder(x, lengths)

    def encode_pooled(self, x: torch.Tensor, lengths: torch.Tensor) -> dict[str, torch.Tensor]:
        outputs = self.encoder(x, lengths)
        pooled = masked_mean_pool(outputs["hidden"], outputs["token_lengths"])
        z = F.normalize(self.segment_head(pooled), dim=-1)
        return {**outputs, "pooled": pooled, "z": z}


def compute_future_infonce_metrics(
    model: ContrastiveSSLModel,
    batch,
    *,
    temperature: float,
    horizons: tuple[int, ...],
    device: torch.device,
):
    outputs = model.encode_sequence(batch["x"].to(device), batch["lengths"].to(device))
    anchor_z = F.normalize(model.anchor_head(outputs["hidden"]), dim=-1)
    future_z = F.normalize(model.future_head(outputs["hidden"]), dim=-1)

    horizon_losses = {}
    horizon_top1 = {}
    loss_terms = []
    top1_terms = []
    positive_pairs = 0

    for horizon in horizons:
        q_list = []
        k_list = []
        for idx, length in enumerate(outputs["token_lengths"].tolist()):
            usable = int(length) - int(horizon)
            if usable <= 0:
                continue
            q_list.append(anchor_z[idx, :usable])
            k_list.append(future_z[idx, horizon:length])
        if not q_list:
            continue

        q = torch.cat(q_list, dim=0)
        k = torch.cat(k_list, dim=0)
        logits = q @ k.T / float(temperature)
        labels = torch.arange(q.shape[0], device=logits.device)
        loss = F.cross_entropy(logits, labels)
        top1 = (logits.argmax(dim=1) == labels).float().mean()

        horizon_losses[int(horizon)] = float(loss.detach().cpu().item())
        horizon_top1[int(horizon)] = float(top1.detach().cpu().item())
        loss_terms.append(loss)
        top1_terms.append(top1)
        positive_pairs += int(q.shape[0])

    if not loss_terms:
        raise ValueError("No valid future InfoNCE positive pairs remain. Reduce patching or horizon values.")

    return {
        "loss": torch.stack(loss_terms).mean(),
        "top1": torch.stack(top1_terms).mean(),
        "per_horizon_losses": horizon_losses,
        "per_horizon_top1": horizon_top1,
        "positive_pairs": positive_pairs,
    }


def augment_segment(x_seq: torch.Tensor, feature_mask: torch.Tensor, cfg: dict) -> torch.Tensor:
    x_aug = x_seq.clone()
    present_idx = torch.nonzero(feature_mask.bool(), as_tuple=False).squeeze(1)
    if present_idx.numel() == 0:
        return x_aug

    present = x_aug[:, present_idx]
    present = present + torch.randn_like(present) * float(cfg["noise_std"])

    scale = 1.0 + torch.randn(present.shape[1], dtype=present.dtype) * float(cfg["scale_jitter"])
    offset = torch.randn(present.shape[1], dtype=present.dtype) * float(cfg["offset_jitter"])
    present = present * scale.unsqueeze(0) + offset.unsqueeze(0)

    time_mask_frac = float(cfg["time_mask_frac"])
    if time_mask_frac > 0.0 and present.shape[0] > 1:
        width = max(1, int(round(time_mask_frac * present.shape[0])))
        width = min(width, present.shape[0])
        start = random.randrange(present.shape[0] - width + 1)
        present[start : start + width] = 0.0

    dropout_prob = float(cfg["channel_dropout_prob"])
    if dropout_prob > 0.0 and present.shape[1] > 1:
        keep = torch.rand(present.shape[1]) > dropout_prob
        if not bool(keep.any()):
            keep[random.randrange(present.shape[1])] = True
        present = present * keep.to(present.dtype).unsqueeze(0)

    x_aug[:, present_idx] = present.clamp(min=-float(cfg["clip_value"]), max=float(cfg["clip_value"]))
    return x_aug


def build_augmented_views(batch, cfg: dict) -> tuple[torch.Tensor, torch.Tensor]:
    view1 = []
    view2 = []
    for x_seq, feature_mask in zip(batch["x"], batch["feature_mask"]):
        view1.append(augment_segment(x_seq, feature_mask, cfg))
        view2.append(augment_segment(x_seq, feature_mask, cfg))
    return torch.stack(view1, dim=0), torch.stack(view2, dim=0)


def compute_augment_infonce_metrics(
    model: ContrastiveSSLModel,
    batch,
    *,
    temperature: float,
    augment_cfg: dict,
    device: torch.device,
):
    view1, view2 = build_augmented_views(batch, augment_cfg)
    lengths = batch["lengths"].to(device)
    out1 = model.encode_pooled(view1.to(device), lengths)
    out2 = model.encode_pooled(view2.to(device), lengths)
    z1 = out1["z"]
    z2 = out2["z"]

    logits12 = z1 @ z2.T / float(temperature)
    logits21 = z2 @ z1.T / float(temperature)
    labels = torch.arange(z1.shape[0], device=z1.device)
    loss12 = F.cross_entropy(logits12, labels)
    loss21 = F.cross_entropy(logits21, labels)
    top1_12 = (logits12.argmax(dim=1) == labels).float().mean()
    top1_21 = (logits21.argmax(dim=1) == labels).float().mean()

    return {
        "loss": 0.5 * (loss12 + loss21),
        "top1": 0.5 * (top1_12 + top1_21),
        "positive_pairs": int(z1.shape[0]),
        "mean_abs_view_delta": float((view1 - view2).abs().mean().item()),
    }


def compute_objective_metrics(
    model: ContrastiveSSLModel,
    batch,
    *,
    objective_mode: str,
    device: torch.device,
    temperature: float,
    horizons: tuple[int, ...],
    augment_cfg: dict,
):
    if objective_mode == "future_infonce":
        return compute_future_infonce_metrics(
            model,
            batch,
            temperature=temperature,
            horizons=horizons,
            device=device,
        )
    if objective_mode == "augment_infonce":
        return compute_augment_infonce_metrics(
            model,
            batch,
            temperature=temperature,
            augment_cfg=augment_cfg,
            device=device,
        )
    raise ValueError(f"Unknown objective_mode: {objective_mode}")


smoke_sampler = build_segment_sampler(
    "train",
    batch_size=8,
    seed=SEED,
    segment_bins=SEGMENT_BINS,
    dataset_weight_alpha=DATASET_WEIGHT_ALPHA,
    examples_per_shard=EXAMPLES_PER_SHARD,
)
smoke_batch = smoke_sampler.sample_batch()
smoke_model = ContrastiveSSLModel().to(DEVICE)
smoke_optimizer = torch.optim.AdamW(smoke_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

for smoke_mode in ("future_infonce", "augment_infonce"):
    smoke_model.train()
    smoke_optimizer.zero_grad(set_to_none=True)
    metrics = compute_objective_metrics(
        smoke_model,
        smoke_batch,
        objective_mode=smoke_mode,
        device=DEVICE,
        temperature=TEMPERATURE,
        horizons=FUTURE_HORIZONS,
        augment_cfg=AUGMENT_CFG,
    )
    loss = metrics["loss"]
    loss.backward()
    grad_norm = torch.nn.utils.clip_grad_norm_(smoke_model.parameters(), 1.0)
    smoke_optimizer.step()

    print(f"smoke_mode={smoke_mode}")
    print(" - loss:", float(loss.detach().cpu().item()))
    print(" - top1:", float(metrics["top1"].detach().cpu().item()))
    print(" - positive_pairs:", int(metrics.get("positive_pairs", 0)))
    print(" - grad_norm:", float(grad_norm))
    if "per_horizon_losses" in metrics:
        print(" - per_horizon_losses:", metrics["per_horizon_losses"])
        print(" - per_horizon_top1:", metrics["per_horizon_top1"])
    if "mean_abs_view_delta" in metrics:
        print(" - mean_abs_view_delta:", float(metrics["mean_abs_view_delta"]))

print("\nDefault run mode:", OBJECTIVE_MODE)


In [7]:
# Train the selected objective and log train / validation metrics.
import json
import time
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path


def summarize_metrics(metrics: dict) -> dict:
    summary = {
        "loss": float(metrics["loss"].detach().cpu().item()),
        "top1": float(metrics["top1"].detach().cpu().item()),
        "positive_pairs": int(metrics.get("positive_pairs", 0)),
    }
    if "per_horizon_losses" in metrics:
        summary["per_horizon_losses"] = {str(k): float(v) for k, v in metrics["per_horizon_losses"].items()}
    if "per_horizon_top1" in metrics:
        summary["per_horizon_top1"] = {str(k): float(v) for k, v in metrics["per_horizon_top1"].items()}
    if "mean_abs_view_delta" in metrics:
        summary["mean_abs_view_delta"] = float(metrics["mean_abs_view_delta"])
    return summary


def evaluate_model(
    model: ContrastiveSSLModel,
    sampler,
    *,
    objective_mode: str,
    num_batches: int,
    device: torch.device,
) -> dict | None:
    if sampler is None:
        return None

    was_training = model.training
    model.eval()

    losses = []
    top1_values = []
    positive_pairs = []
    per_horizon_losses = defaultdict(list)
    per_horizon_top1 = defaultdict(list)
    view_deltas = []

    with torch.no_grad():
        for _ in range(num_batches):
            batch = sampler.sample_batch()
            metrics = compute_objective_metrics(
                model,
                batch,
                objective_mode=objective_mode,
                device=device,
                temperature=TEMPERATURE,
                horizons=FUTURE_HORIZONS,
                augment_cfg=AUGMENT_CFG,
            )
            summary = summarize_metrics(metrics)
            losses.append(summary["loss"])
            top1_values.append(summary["top1"])
            positive_pairs.append(summary["positive_pairs"])
            for horizon, value in summary.get("per_horizon_losses", {}).items():
                per_horizon_losses[horizon].append(float(value))
            for horizon, value in summary.get("per_horizon_top1", {}).items():
                per_horizon_top1[horizon].append(float(value))
            if "mean_abs_view_delta" in summary:
                view_deltas.append(float(summary["mean_abs_view_delta"]))

    if was_training:
        model.train()

    result = {
        "loss": float(np.mean(losses)),
        "top1": float(np.mean(top1_values)),
        "positive_pairs": int(np.sum(positive_pairs)),
    }
    if per_horizon_losses:
        result["per_horizon_losses"] = {key: float(np.mean(values)) for key, values in per_horizon_losses.items()}
    if per_horizon_top1:
        result["per_horizon_top1"] = {key: float(np.mean(values)) for key, values in per_horizon_top1.items()}
    if view_deltas:
        result["mean_abs_view_delta"] = float(np.mean(view_deltas))
    return result


TRAIN_SAMPLER = build_segment_sampler(
    "train",
    batch_size=BATCH_SIZE,
    seed=SEED,
    segment_bins=SEGMENT_BINS,
    dataset_weight_alpha=DATASET_WEIGHT_ALPHA,
    examples_per_shard=EXAMPLES_PER_SHARD,
)
try:
    VAL_SAMPLER = build_segment_sampler(
        "val",
        batch_size=BATCH_SIZE,
        seed=SEED + 101,
        segment_bins=SEGMENT_BINS,
        dataset_weight_alpha=DATASET_WEIGHT_ALPHA,
        examples_per_shard=EXAMPLES_PER_SHARD,
    )
except RuntimeError:
    VAL_SAMPLER = None

model = ContrastiveSSLModel().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

run_name = (
    f"colab_s5_{OBJECTIVE_MODE}_seg{SEGMENT_BINS}_"
    f"{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"
)
run_dir = OUTPUT_ROOT / run_name
run_dir.mkdir(parents=True, exist_ok=True)
progress_path = run_dir / "progress.jsonl"
checkpoint_path = run_dir / "checkpoint_final.pt"
plot_loss_path = run_dir / "loss_curve.png"
plot_top1_path = run_dir / "retrieval_curve.png"

config = {
    "seed": SEED,
    "objective_mode": OBJECTIVE_MODE,
    "segment_bins": SEGMENT_BINS,
    "future_horizons": list(FUTURE_HORIZONS),
    "patch_size": PATCH_SIZE,
    "patch_stride": PATCH_STRIDE,
    "hidden_size": HIDDEN_SIZE,
    "s5_state_size": S5_STATE_SIZE,
    "num_layers": NUM_LAYERS,
    "dropout": DROPOUT,
    "batch_size": BATCH_SIZE,
    "num_steps": NUM_STEPS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "temperature": TEMPERATURE,
    "val_every": VAL_EVERY,
    "val_batches": VAL_BATCHES,
    "dataset_weight_alpha": DATASET_WEIGHT_ALPHA,
    "examples_per_shard": EXAMPLES_PER_SHARD,
    "augment_cfg": AUGMENT_CFG,
    "normalize_impl_version": NORMALIZE_IMPL_VERSION,
    "normalize_context_bins": NORMALIZE_CONTEXT_BINS,
    "post_proj_norm": POST_PROJ_NORM,
    "has_val_datasets": HAS_VAL_DATASETS,
    "session_split_summary": session_split_summary,
    "cache_root": str(CACHE_ROOT),
    "output_root": str(OUTPUT_ROOT),
}
(run_dir / "config.json").write_text(json.dumps(config, indent=2))

best_score = float("inf")
best_state = None
train_history = []
val_history = []
dataset_counter = Counter()
start_time = time.time()

for step in range(1, NUM_STEPS + 1):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    t0 = time.time()
    batch = TRAIN_SAMPLER.sample_batch()
    sample_seconds = time.time() - t0
    dataset_mix = dict(Counter(batch["datasets"]))
    dataset_counter.update(batch["datasets"])

    sync_device(DEVICE)
    t1 = time.time()
    metrics = compute_objective_metrics(
        model,
        batch,
        objective_mode=OBJECTIVE_MODE,
        device=DEVICE,
        temperature=TEMPERATURE,
        horizons=FUTURE_HORIZONS,
        augment_cfg=AUGMENT_CFG,
    )
    loss = metrics["loss"]
    loss.backward()
    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    sync_device(DEVICE)
    model_seconds = time.time() - t1

    summary = summarize_metrics(metrics)
    train_record = {
        "event": "train",
        "step": step,
        "elapsed_seconds": float(time.time() - start_time),
        "sample_seconds": float(sample_seconds),
        "model_seconds": float(model_seconds),
        "grad_norm": float(grad_norm),
        "cached_shards": int(SHARD_STORE.summary()["cached_shards"]),
        "cached_gb": float(SHARD_STORE.summary()["cached_gb"]),
        "dataset_mix": dataset_mix,
        **summary,
    }
    train_history.append(train_record)
    with progress_path.open("a") as handle:
        handle.write(json.dumps(train_record) + "\n")

    if VAL_SAMPLER is None and summary["loss"] < best_score:
        best_score = summary["loss"]
        best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}

    if step == 1 or step % LOG_EVERY == 0:
        print(
            f"step={step:03d} train_loss={summary['loss']:.4f} train_top1={summary['top1']:.4f} "
            f"grad_norm={float(grad_norm):.4f} sample_s={sample_seconds:.2f} model_s={model_seconds:.2f}"
        )

    if VAL_SAMPLER is not None and (step % VAL_EVERY == 0 or step == NUM_STEPS):
        val_result = evaluate_model(
            model,
            VAL_SAMPLER,
            objective_mode=OBJECTIVE_MODE,
            num_batches=VAL_BATCHES,
            device=DEVICE,
        )
        val_record = {
            "event": "val",
            "step": step,
            "elapsed_seconds": float(time.time() - start_time),
            **val_result,
        }
        val_history.append(val_record)
        with progress_path.open("a") as handle:
            handle.write(json.dumps(val_record) + "\n")
        print(
            f"step={step:03d} val_loss={val_result['loss']:.4f} val_top1={val_result['top1']:.4f} "
            f"positive_pairs={val_result['positive_pairs']}"
        )
        if val_result["loss"] < best_score:
            best_score = val_result["loss"]
            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}

if best_state is not None:
    model.load_state_dict(best_state)
else:
    best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}

torch.save(
    {
        "model_state": model.state_dict(),
        "config": config,
        "best_score": best_score,
        "train_history": train_history,
        "val_history": val_history,
        "dataset_counts": dict(dataset_counter),
    },
    checkpoint_path,
)

print("run_dir:", run_dir)
print("progress_path:", progress_path)
print("checkpoint_path:", checkpoint_path)
print("best_score:", best_score)
print("dataset_counts:", dict(dataset_counter))


In [ ]:
# Plot train / validation loss and retrieval accuracy.
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

if "progress_path" not in globals():
    raise RuntimeError("Run the training cell first so progress_path is defined.")

records = [json.loads(line) for line in progress_path.read_text().splitlines() if line.strip()]
train_records = [record for record in records if record.get("event") == "train"]
val_records = [record for record in records if record.get("event") == "val"]

train_steps = np.array([record["step"] for record in train_records], dtype=np.int64)
train_losses = np.array([record["loss"] for record in train_records], dtype=np.float32)
train_top1 = np.array([record["top1"] for record in train_records], dtype=np.float32)

plt.figure(figsize=(8, 4))
plt.plot(train_steps, train_losses, label="train loss", alpha=0.8)
if val_records:
    val_steps = np.array([record["step"] for record in val_records], dtype=np.int64)
    val_losses = np.array([record["loss"] for record in val_records], dtype=np.float32)
    plt.plot(val_steps, val_losses, marker="o", label="val loss")
plt.xlabel("step")
plt.ylabel("loss")
plt.title(f"{OBJECTIVE_MODE} loss")
plt.legend()
plt.tight_layout()
plt.savefig(plot_loss_path, dpi=160)
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(train_steps, train_top1, label="train top1", alpha=0.8)
if val_records:
    val_top1 = np.array([record["top1"] for record in val_records], dtype=np.float32)
    plt.plot(val_steps, val_top1, marker="o", label="val top1")
plt.xlabel("step")
plt.ylabel("retrieval top1")
plt.title(f"{OBJECTIVE_MODE} retrieval accuracy")
plt.legend()
plt.tight_layout()
plt.savefig(plot_top1_path, dpi=160)
plt.show()

print("train_start_loss:", float(train_losses[0]))
print("train_final_loss:", float(train_losses[-1]))
print("train_best_loss:", float(train_losses.min()), "at step", int(train_steps[train_losses.argmin()]))
print("train_final_top1:", float(train_top1[-1]))
if val_records:
    print("val_best_loss:", float(val_losses.min()), "at step", int(val_steps[val_losses.argmin()]))
    print("val_best_top1:", float(val_top1.max()), "at step", int(val_steps[val_top1.argmax()]))
print("plot_loss_path:", plot_loss_path)
print("plot_top1_path:", plot_top1_path)


## Benchmark-lite Downstream Phoneme Probe

This section runs a cheap held-out Brain2Text25 causal phoneme probe on the current SSL encoder.
It follows the staged BIT logic: evaluate encoder quality with phoneme CTC before any sentence-level decoding.


In [ ]:
# Benchmark-lite downstream phoneme probe config.
DOWNSTREAM_PROBE_CONFIG = {
    "enabled": True,
    "comparison_mode": "ssl_only",
    "session_limit": 4,
    "target_session_count": 1,
    "adaptation_regime": "A",
    "probe_batch_size": 8,
    "probe_budget_seconds": 120,
    "max_probe_steps": 200,
    "progress_every_steps": 25,
    "checkpoint_source": "in_memory_then_checkpoint",
}
DOWNSTREAM_PROBE_PROGRESS_EVERY_SECONDS = 15.0
DOWNSTREAM_PROBE_SUMMARY_BASENAME = "downstream_probe_summary.json"

assert DOWNSTREAM_PROBE_CONFIG["comparison_mode"] == "ssl_only"
assert DOWNSTREAM_PROBE_CONFIG["adaptation_regime"] == "A"
assert DOWNSTREAM_PROBE_CONFIG["target_session_count"] == 1

print("DOWNSTREAM_PROBE_CONFIG:")
for key, value in DOWNSTREAM_PROBE_CONFIG.items():
    print(f" - {key}: {value}")
print(" - progress_every_seconds:", DOWNSTREAM_PROBE_PROGRESS_EVERY_SECONDS)
print(" - summary_basename:", DOWNSTREAM_PROBE_SUMMARY_BASENAME)


In [ ]:
# Import and validate benchmark-lite downstream probe helpers.
from pathlib import Path

from data import (
    load_b2t25_canonical_inventory,
    load_probe_manifest,
    load_probe_metadata,
    partition_probe_records,
    session_ids_from_cache_split,
    split_latest_sessions,
)
from train import RunConfig, train_probe_for_session

CANONICAL_B2T25_ROOT = Path(CACHE_ROOT) / "brain2text25"
CANONICAL_B2T25_MANIFEST_PATH = CANONICAL_B2T25_ROOT / "manifest.jsonl"
CANONICAL_B2T25_METADATA_PATH = CANONICAL_B2T25_ROOT / "metadata.json"

if not CANONICAL_B2T25_MANIFEST_PATH.exists() or not CANONICAL_B2T25_METADATA_PATH.exists():
    raise FileNotFoundError(
        "Canonical Brain2Text25 cache manifest / metadata is missing from the mounted cache. "
        f"Expected {CANONICAL_B2T25_MANIFEST_PATH} and {CANONICAL_B2T25_METADATA_PATH}."
    )

downstream_probe_metadata_preview = load_probe_metadata(CANONICAL_B2T25_METADATA_PATH)
if "phoneme_vocabulary" not in downstream_probe_metadata_preview:
    raise KeyError("Canonical Brain2Text25 metadata is missing 'phoneme_vocabulary'.")

downstream_probe_inventory = load_b2t25_canonical_inventory(CANONICAL_B2T25_ROOT)
downstream_probe_eligible_entries = [entry for entry in downstream_probe_inventory if entry.has_tx and entry.has_sbp]
if len(downstream_probe_eligible_entries) < DOWNSTREAM_PROBE_CONFIG["session_limit"]:
    raise ValueError(
        "Not enough matched Brain2Text25 sessions are available for the benchmark-lite probe. "
        f"Need at least {DOWNSTREAM_PROBE_CONFIG['session_limit']}, found {len(downstream_probe_eligible_entries)}."
    )

downstream_probe_split_preview = split_latest_sessions(
    downstream_probe_eligible_entries,
    session_limit=DOWNSTREAM_PROBE_CONFIG["session_limit"],
    val_session_count=DOWNSTREAM_PROBE_CONFIG["target_session_count"],
)

print("Canonical Brain2Text25 probe assets:")
print(" - root:", CANONICAL_B2T25_ROOT)
print(" - manifest:", CANONICAL_B2T25_MANIFEST_PATH)
print(" - metadata:", CANONICAL_B2T25_METADATA_PATH)
print(" - eligible_sessions:", len(downstream_probe_eligible_entries))
print(" - preview_source_sessions:", [entry.session_base for entry in downstream_probe_split_preview.train])
print(" - preview_target_sessions:", [entry.session_base for entry in downstream_probe_split_preview.val])


In [ ]:
# Recover a notebook encoder and wrap it for the benchmark-lite probe.
import copy
from datetime import datetime, timezone
from pathlib import Path
from types import SimpleNamespace

import torch
import torch.nn as nn

if "DEVICE" not in globals():
    raise RuntimeError("Run the encoder definition cell before the downstream probe cells so DEVICE and S5ContrastiveEncoder exist.")


class NotebookProbeEncoderAdapter(nn.Module):
    def __init__(self, encoder: nn.Module):
        super().__init__()
        self.encoder = encoder
        self.input_dim = int(getattr(encoder, "input_dim"))
        self.hidden_size = int(getattr(encoder, "hidden_size"))

    def encode(
        self,
        x: torch.Tensor,
        input_lengths: torch.Tensor,
        session_ids: list[str],
        *,
        use_source_affines: bool,
        target_affines=None,
    ):
        del use_source_affines
        if target_affines is not None:
            x = target_affines(x, session_ids)
        outputs = self.encoder(x, input_lengths)
        return SimpleNamespace(
            hidden=outputs["hidden"],
            token_lengths=outputs["token_lengths"],
            tokens=outputs["tokens"],
        )


def _resolve_notebook_checkpoint_path() -> Path | None:
    if "checkpoint_path" in globals():
        candidate = Path(checkpoint_path)
        if candidate.exists():
            return candidate
    candidates = sorted(
        OUTPUT_ROOT.glob("colab_s5_*/checkpoint_final.pt"),
        key=lambda candidate: candidate.stat().st_mtime,
    )
    return candidates[-1] if candidates else None


def _recover_encoder_from_notebook_checkpoint(path: Path):
    payload = torch.load(path, map_location="cpu")
    checkpoint_cfg = payload.get("config", {})
    required_keys = [
        "patch_size",
        "patch_stride",
        "hidden_size",
        "s5_state_size",
        "num_layers",
        "dropout",
    ]
    missing_keys = [key for key in required_keys if key not in checkpoint_cfg]
    if missing_keys:
        raise KeyError(f"Checkpoint config is missing keys needed for encoder recovery: {missing_keys}")

    recovered_encoder = S5ContrastiveEncoder(
        input_dim=FULL_DIM,
        hidden_size=int(checkpoint_cfg["hidden_size"]),
        s5_state_size=int(checkpoint_cfg["s5_state_size"]),
        num_layers=int(checkpoint_cfg["num_layers"]),
        dropout=float(checkpoint_cfg["dropout"]),
        patch_size=int(checkpoint_cfg["patch_size"]),
        patch_stride=int(checkpoint_cfg["patch_stride"]),
        post_proj_norm=str(checkpoint_cfg.get("post_proj_norm", "rms")),
    )
    model_state = payload.get("model_state")
    if model_state is None:
        raise KeyError("Notebook checkpoint is missing 'model_state'.")

    encoder_state = {
        key.split("encoder.", 1)[1]: value
        for key, value in model_state.items()
        if key.startswith("encoder.")
    }
    if not encoder_state:
        raise KeyError("Notebook checkpoint does not contain encoder weights.")
    recovered_encoder.load_state_dict(encoder_state)
    return recovered_encoder, checkpoint_cfg


def _resolve_downstream_probe_run_dir(checkpoint_candidate: Path | None) -> Path:
    if "run_dir" in globals():
        candidate = Path(run_dir)
        if candidate.exists():
            return candidate
    if checkpoint_candidate is not None:
        return checkpoint_candidate.parent
    fallback = OUTPUT_ROOT / f"colab_s5_downstream_probe_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"
    fallback.mkdir(parents=True, exist_ok=True)
    return fallback


DOWNSTREAM_PROBE_CHECKPOINT_PATH = _resolve_notebook_checkpoint_path()
DOWNSTREAM_PROBE_RUN_DIR = _resolve_downstream_probe_run_dir(DOWNSTREAM_PROBE_CHECKPOINT_PATH)
DOWNSTREAM_PROBE_CHECKPOINT_CONFIG = {}

if (
    DOWNSTREAM_PROBE_CONFIG["checkpoint_source"] == "in_memory_then_checkpoint"
    and "model" in globals()
    and hasattr(model, "encoder")
):
    downstream_probe_base_encoder = copy.deepcopy(model.encoder).cpu()
    DOWNSTREAM_PROBE_ENCODER_SOURCE = "in_memory"
    DOWNSTREAM_PROBE_CHECKPOINT_CONFIG = {
        "patch_size": PATCH_SIZE,
        "patch_stride": PATCH_STRIDE,
        "hidden_size": HIDDEN_SIZE,
        "s5_state_size": S5_STATE_SIZE,
        "num_layers": NUM_LAYERS,
        "dropout": DROPOUT,
        "post_proj_norm": POST_PROJ_NORM,
    }
elif DOWNSTREAM_PROBE_CHECKPOINT_PATH is not None:
    downstream_probe_base_encoder, DOWNSTREAM_PROBE_CHECKPOINT_CONFIG = _recover_encoder_from_notebook_checkpoint(
        DOWNSTREAM_PROBE_CHECKPOINT_PATH
    )
    downstream_probe_base_encoder = downstream_probe_base_encoder.cpu()
    DOWNSTREAM_PROBE_ENCODER_SOURCE = "checkpoint"
else:
    raise RuntimeError(
        "No in-memory encoder is available and no checkpoint_final.pt was found under OUTPUT_ROOT. "
        "Run the training cell first or make the checkpoint available under the notebook output root."
    )

DOWNSTREAM_PROBE_ENCODER = NotebookProbeEncoderAdapter(downstream_probe_base_encoder)

print("DOWNSTREAM_PROBE_ENCODER_SOURCE:", DOWNSTREAM_PROBE_ENCODER_SOURCE)
print("DOWNSTREAM_PROBE_RUN_DIR:", DOWNSTREAM_PROBE_RUN_DIR)
print("DOWNSTREAM_PROBE_CHECKPOINT_PATH:", DOWNSTREAM_PROBE_CHECKPOINT_PATH)
print("DOWNSTREAM_PROBE_ENCODER:", DOWNSTREAM_PROBE_ENCODER)


In [ ]:
# Run the benchmark-lite held-out Brain2Text25 phoneme probe.
import json
import time
from pathlib import Path

import pandas as pd

if not DOWNSTREAM_PROBE_CONFIG["enabled"]:
    raise RuntimeError("DOWNSTREAM_PROBE_CONFIG['enabled'] is False. Enable it before running the downstream probe.")
if DOWNSTREAM_PROBE_CONFIG["comparison_mode"] != "ssl_only":
    raise NotImplementedError(
        f"comparison_mode={DOWNSTREAM_PROBE_CONFIG['comparison_mode']} is not implemented in this notebook pass."
    )

DOWNSTREAM_PROBE_RUN_DIR.mkdir(parents=True, exist_ok=True)
downstream_probe_progress_path = DOWNSTREAM_PROBE_RUN_DIR / "downstream_probe.progress.jsonl"
if downstream_probe_progress_path.exists():
    downstream_probe_progress_path.unlink()

downstream_probe_inventory = load_b2t25_canonical_inventory(CANONICAL_B2T25_ROOT)
downstream_probe_eligible_entries = [entry for entry in downstream_probe_inventory if entry.has_tx and entry.has_sbp]
downstream_probe_split = split_latest_sessions(
    downstream_probe_eligible_entries,
    session_limit=DOWNSTREAM_PROBE_CONFIG["session_limit"],
    val_session_count=DOWNSTREAM_PROBE_CONFIG["target_session_count"],
)
downstream_probe_source_session_ids, downstream_probe_target_session_ids = session_ids_from_cache_split(downstream_probe_split)

if len(downstream_probe_target_session_ids) != 1:
    raise ValueError(
        "The benchmark-lite notebook probe expects exactly one held-out target session. "
        f"Found {len(downstream_probe_target_session_ids)} target sessions."
    )

downstream_probe_manifest_rows = load_probe_manifest(CANONICAL_B2T25_MANIFEST_PATH)
downstream_probe_metadata = load_probe_metadata(CANONICAL_B2T25_METADATA_PATH)
downstream_probe_partitions = partition_probe_records(
    downstream_probe_manifest_rows,
    source_session_ids=downstream_probe_source_session_ids,
    target_session_ids=downstream_probe_target_session_ids,
)

downstream_probe_target_session_id = downstream_probe_target_session_ids[0]
downstream_probe_target_train_rows = downstream_probe_partitions.target_train_by_session[downstream_probe_target_session_id]
downstream_probe_target_val_rows = downstream_probe_partitions.target_val_by_session[downstream_probe_target_session_id]

if len(downstream_probe_target_train_rows) == 0 or len(downstream_probe_target_val_rows) == 0:
    raise ValueError(
        "The held-out target session does not have both train and val examples with phoneme labels. "
        f"train={len(downstream_probe_target_train_rows)} val={len(downstream_probe_target_val_rows)}"
    )

downstream_probe_vocab = downstream_probe_metadata["phoneme_vocabulary"]
downstream_probe_config_obj = RunConfig(
    profile="colab_cuda",
    dataset_family="brain2text25",
    backbone="s5",
    objective_family="future_prediction",
    adaptation_regime=DOWNSTREAM_PROBE_CONFIG["adaptation_regime"],
    patch_size=int(DOWNSTREAM_PROBE_CHECKPOINT_CONFIG.get("patch_size", PATCH_SIZE)),
    patch_stride=int(DOWNSTREAM_PROBE_CHECKPOINT_CONFIG.get("patch_stride", PATCH_STRIDE)),
    standardize_scope="session",
    post_proj_norm=str(DOWNSTREAM_PROBE_CHECKPOINT_CONFIG.get("post_proj_norm", POST_PROJ_NORM)),
    hidden_size=int(DOWNSTREAM_PROBE_CHECKPOINT_CONFIG.get("hidden_size", HIDDEN_SIZE)),
    s5_state_size=int(DOWNSTREAM_PROBE_CHECKPOINT_CONFIG.get("s5_state_size", S5_STATE_SIZE)),
    num_layers=int(DOWNSTREAM_PROBE_CHECKPOINT_CONFIG.get("num_layers", NUM_LAYERS)),
    dropout=float(DOWNSTREAM_PROBE_CHECKPOINT_CONFIG.get("dropout", DROPOUT)),
    probe_batch_size=int(DOWNSTREAM_PROBE_CONFIG["probe_batch_size"]),
    probe_budget_seconds=int(DOWNSTREAM_PROBE_CONFIG["probe_budget_seconds"]),
    max_probe_steps=int(DOWNSTREAM_PROBE_CONFIG["max_probe_steps"]),
    progress_every_steps=int(DOWNSTREAM_PROBE_CONFIG["progress_every_steps"]),
    progress_every_seconds=float(DOWNSTREAM_PROBE_PROGRESS_EVERY_SECONDS),
)

downstream_probe_start_time = time.time()
downstream_probe_val_bpphone, downstream_probe_steps = train_probe_for_session(
    pretrained_encoder=DOWNSTREAM_PROBE_ENCODER,
    session_id=downstream_probe_target_session_id,
    train_rows=downstream_probe_target_train_rows,
    val_rows=downstream_probe_target_val_rows,
    probe_vocab_size=int(downstream_probe_vocab["num_classes"]),
    blank_index=int(downstream_probe_vocab["blank_index"]),
    config=downstream_probe_config_obj,
    device=DEVICE,
    budget_seconds=float(DOWNSTREAM_PROBE_CONFIG["probe_budget_seconds"]),
    progress_log_path=downstream_probe_progress_path,
)
downstream_probe_elapsed_seconds = time.time() - downstream_probe_start_time

downstream_probe_summary = {
    "comparison_mode": DOWNSTREAM_PROBE_CONFIG["comparison_mode"],
    "adaptation_regime": DOWNSTREAM_PROBE_CONFIG["adaptation_regime"],
    "session_limit": int(DOWNSTREAM_PROBE_CONFIG["session_limit"]),
    "target_session_count": int(DOWNSTREAM_PROBE_CONFIG["target_session_count"]),
    "heldout_target_session_id": downstream_probe_target_session_id,
    "source_session_ids": list(downstream_probe_source_session_ids),
    "target_session_ids": list(downstream_probe_target_session_ids),
    "selected_session_bases": [
        entry.session_base for entry in downstream_probe_split.train + downstream_probe_split.val
    ],
    "target_train_examples": len(downstream_probe_target_train_rows),
    "target_val_examples": len(downstream_probe_target_val_rows),
    "val_bpphone": float(downstream_probe_val_bpphone),
    "probe_steps": int(downstream_probe_steps),
    "probe_elapsed_seconds": float(downstream_probe_elapsed_seconds),
    "checkpoint_source_used": DOWNSTREAM_PROBE_ENCODER_SOURCE,
    "checkpoint_path": str(DOWNSTREAM_PROBE_CHECKPOINT_PATH) if DOWNSTREAM_PROBE_CHECKPOINT_PATH is not None else None,
    "run_dir": str(DOWNSTREAM_PROBE_RUN_DIR),
    "progress_log_path": str(downstream_probe_progress_path),
}

downstream_probe_summary_path = DOWNSTREAM_PROBE_RUN_DIR / DOWNSTREAM_PROBE_SUMMARY_BASENAME
downstream_probe_summary_path.write_text(json.dumps(downstream_probe_summary, indent=2))

display(pd.DataFrame([downstream_probe_summary]))
print("downstream_probe_summary_path:", downstream_probe_summary_path)
print("downstream_probe_progress_path:", downstream_probe_progress_path)


In [8]:
# Estimate how much of the train corpus we covered.
# Uses notebook globals already created in the data-loading cell.

import math
import numpy as np
import pandas as pd

if "split_rows_by_dataset" not in globals():
    raise RuntimeError("Run the data-loading cell first.")
if "TRAIN_SAMPLER" not in globals():
    raise RuntimeError("Run the training setup cell first.")

steps_run = 200
batch_size = BATCH_SIZE
segment_bins = SEGMENT_BINS
sampled_segments = steps_run * batch_size
sampled_bins = sampled_segments * segment_bins

plan = TRAIN_SAMPLER.plan
dataset_probs = {
    dataset: float(prob)
    for dataset, prob in zip(plan.dataset_names, plan.dataset_probs)
}

rows = []
total_valid_windows = 0

for dataset in pretrain_datasets:
    train_rows = split_rows_by_dataset["train"][dataset]
    valid_windows = sum(max(0, row.n_time_bins - segment_bins + 1) for row in train_rows)
    eligible_examples = sum(1 for row in train_rows if row.n_time_bins >= segment_bins)
    expected_draws = sampled_segments * dataset_probs.get(dataset, 0.0)

    approx_epoch_fraction = (
        expected_draws / valid_windows if valid_windows > 0 else np.nan
    )

    # Approximate unique-window coverage if draws were iid over valid windows.
    approx_unique_fraction = (
        1.0 - math.exp(-expected_draws / valid_windows)
        if valid_windows > 0 else np.nan
    )

    rows.append({
        "dataset": dataset,
        "train_examples": len(train_rows),
        "eligible_examples": eligible_examples,
        "valid_windows": int(valid_windows),
        "dataset_prob": dataset_probs.get(dataset, 0.0),
        "expected_segment_draws_in_run": float(expected_draws),
        "approx_epoch_fraction": float(approx_epoch_fraction) if valid_windows > 0 else np.nan,
        "approx_unique_window_coverage": float(approx_unique_fraction) if valid_windows > 0 else np.nan,
    })

    total_valid_windows += valid_windows

df = pd.DataFrame(rows).sort_values("valid_windows", ascending=False).reset_index(drop=True)
display(df)

print("steps_run:", steps_run)
print("batch_size:", batch_size)
print("sampled_segments:", sampled_segments)
print("sampled_bins:", sampled_bins)
print("total_valid_windows_train:", total_valid_windows)

if total_valid_windows > 0:
    print("raw_window_epoch_equivalent:", sampled_segments / total_valid_windows)
    print("approx_global_unique_window_coverage:", 1.0 - math.exp(-sampled_segments / total_valid_windows))


In [9]:
from pathlib import Path
import numpy as np
import pandas as pd

cache_root = Path(CACHE_ROOT)

shard_dirs = []
for time_offsets_path in cache_root.rglob("time_offsets.npy"):
    shard_dir = time_offsets_path.parent
    total_bytes = sum(
        p.stat().st_size
        for p in shard_dir.iterdir()
        if p.is_file()
    )
    shard_dirs.append({
        "shard_dir": str(shard_dir.relative_to(cache_root)),
        "total_bytes": total_bytes,
        "num_files": sum(1 for p in shard_dir.iterdir() if p.is_file()),
    })

df = pd.DataFrame(shard_dirs)

if len(df) == 0:
    raise RuntimeError("No shard directories found via time_offsets.npy")

df["mb"] = df["total_bytes"] / (1024 ** 2)

display(df.describe())

print("num_shards:", len(df))
print("total_gb:", df["total_bytes"].sum() / (1024 ** 3))
print("mean_mb_per_shard:", df["mb"].mean())
print("median_mb_per_shard:", df["mb"].median())
print("p90_mb_per_shard:", df["mb"].quantile(0.9))
print("min_files_per_shard:", df["num_files"].min())
print("max_files_per_shard:", df["num_files"].max())
